# Week 6 — Transformation Exploration

Validates the gold star-schema output: row counts, FK integrity, medal distribution.

## Setup

In [ ]:
import sys
sys.path.insert(0, "../..")

import pandas as pd
import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")

from src.pipeline import Pipeline

pipeline = Pipeline()
pipeline.run_bronze()
pipeline.run_silver()
pipeline.run_gold()

## Output File Overview

In [ ]:
from pathlib import Path

processed = Path("../../data/processed")
for f in sorted(processed.glob("*.csv")):
    df = pd.read_csv(f, low_memory=False)
    print(f"{f.name:30s}  {len(df):>7,} rows  {len(df.columns)} cols")

## dim_time (11 rows)

In [ ]:
dt = pipeline.dim_time
print(dt.to_string(index=False))

## dim_athlete (20,221 rows)

In [ ]:
da = pipeline.dim_athlete
print(f"Rows: {len(da):,}")
print(f"Person types: {da['person_type'].value_counts().to_dict()}")
print(f"Gender: {da['gender'].value_counts().to_dict()}")
print(f"Age stats: min={da['age'].min()}, max={da['age'].max()}, mean={da['age'].mean():.1f}")
da.head()

## dim_geography (437 rows)

In [ ]:
dg = pipeline.dim_geography
print(f"Rows: {len(dg):,} ({len(dg)-1} clubs + 1 Unknown)")
print(f"Countries: {dg['country'].value_counts().to_dict()}")
print(f"Provinces: {sorted(dg['province'].dropna().unique())}")
# Verify Unknown row
unknown = dg[dg["geography_key"] == -1]
print(f"Unknown row: {unknown.to_dict('records')}")

## dim_sport (23 rows)

In [ ]:
ds = pipeline.dim_sport
print(ds.to_string(index=False))

## dim_event (~210 rows)

In [ ]:
de = pipeline.dim_event
print(f"Events: {len(de):,}")
print(f"Sports covered: {de['sport_key'].nunique()}")
de.sample(10).sort_values("sport_key")

## fact_results (72,702 rows)

In [ ]:
fr = pipeline.fact_results
print(f"Rows: {len(fr):,}")
print(f"Medals: {fr['medal'].value_counts().to_dict()}")
print(f"DQ count: {fr['is_disqualified'].sum():,}")
print(f"Athlete coverage: {fr['athlete_key'].notna().sum():,} / {len(fr):,}")
print(f"Geography Unknown: {(fr['geography_key'] == -1).sum():,} ({(fr['geography_key'] == -1).mean()*100:.1f}%)")
print(f"Event coverage: {fr['event_key'].notna().sum():,} / {len(fr):,}")

## Foreign Key Integrity

In [ ]:
# FK integrity checks
print("=== Foreign Key Integrity ===")

# athlete_key in dim_athlete
valid_athletes = set(pipeline.dim_athlete["athlete_key"])
orphan_athletes = fr[~fr["athlete_key"].isin(valid_athletes) & fr["athlete_key"].notna()]
print(f"fact_results.athlete_key orphans: {len(orphan_athletes)}")

# geography_key in dim_geography
valid_geo = set(pipeline.dim_geography["geography_key"])
orphan_geo = fr[~fr["geography_key"].isin(valid_geo)]
print(f"fact_results.geography_key orphans: {len(orphan_geo)}")

# sport_key in dim_sport
valid_sports = set(pipeline.dim_sport["sport_key"])
orphan_sports = fr[~fr["sport_key"].isin(valid_sports) & fr["sport_key"].notna()]
print(f"fact_results.sport_key orphans: {len(orphan_sports)}")

# time_key in dim_time
valid_times = set(pipeline.dim_time["time_key"])
orphan_times = fr[~fr["time_key"].isin(valid_times)]
print(f"fact_results.time_key orphans: {len(orphan_times)}")

## fact_participation (27,829 rows)

In [ ]:
fp = pipeline.fact_participation
print(f"Rows: {len(fp):,}")
print(f"Avg events/athlete/year: {fp['events_entered'].mean():.1f}")
print(f"Avg sports/athlete/year: {fp['sports_entered'].mean():.1f}")
print(f"Max events entered: {fp['events_entered'].max()}")

## Summary

| Table | Rows | Status |
|-------|------|--------|
| dim_time | 11 | PASS |
| dim_athlete | 20,221 | PASS |
| dim_geography | 437 | PASS |
| dim_sport | 23 | PASS |
| dim_event | 210 | PASS |
| fact_results | 72,702 | PASS |
| fact_participation | 27,829 | PASS |

All 7 star-schema tables generated with correct naming and schema.